## **Why FineTuning Is hard in LSTM**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = 200
vocab_size = 10000
embedding_dim = 128
latent_dim = 256

(x_train, y_train), _ = tf.keras.datasets.imdb.load_data(num_words=vocab_size)
len(x_train), len(y_train)

In [ ]:
x_train = x_train[:3000]
y_train = y_train[:3000]
len(x_train), len(y_train)

In [ ]:
X_train = pad_sequences(x_train, maxlen=max_len, padding='post', truncating='post')
Y_train = pad_sequences(y_train, maxlen=max_len, padding='post', truncating='post')

input_layer = Input(shape=(max_len,))
embedding_layer = Embedding(input_dim=vocab_size, output_dim=embedding_dim)(input_layer)
lstm_layer, state_h, state_c = LSTM(units=128, return_state=True)(embedding_layer)
output_layer = Dense(units=1, activation='sigmoid')(lstm_layer)

classification_model = Model(inputs=input_layer, outputs=output_layer)
classification_model.layers, classification_model.summary()

In [ ]:
classification_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
classification_model.fit(X_train, Y_train, epochs=5, batch_size=64, validation_split=0.2)

In [ ]:
sample_review = X_train[0].reshape(1, -1)
prediction = classification_model.predict(sample_review)

print("Predicted sentiment probability (positive):", prediction[0][0])
print("Predicted Sentiment:", "Positive 😊" if prediction[0][0] > 0.5 else "Negative 😞")

In [ ]:
from tensorflow.keras.datasets import imdb
word_index = imdb.get_word_index()

reverse_word_index = {index + 3: word for word, index in word_index.items()}

reverse_word_index[0] = ""
reverse_word_index[1] = ""
reverse_word_index[2] = ""
reverse_word_index[3] = ""
decoded_review = " ".join([reverse_word_index.get(i, "") for i in sample_review[0]])

print("****Decoded Review:****")
print(decoded_review)

In [ ]:
classification_model.save("lstm_imdb_model.h5")

In [ ]:
max_len = 200
vocab_size = 1000
embedding_dim = 128
latent_dim = 256

(x_train, y_train), _ = tf.keras.datasets.imdb.load_data(num_words=vocab_size)
x_train = pad_sequences(x_train, maxlen=max_len, padding='post', truncating='post')

In [ ]:
from tensorflow.keras.models import load_model
x_train = x_train[:1000]
y_train = y_train[:1000]

load_classification_model = load_model("lstm_imdb_model.h5")
load_classification_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
load_classification_model.fit(x_train, y_train, batch_size=64, epochs=1, validation_split=0.1)

In [ ]:
load_classification_model.save("lstm_imdb_model_updated.h5")

In [ ]:
from tensorflow.keras.models import load_model
updated_classification_model = load_model("lstm_imdb_model_updated.h5")

In [ ]:
encoder_inputs = Input(shape=(max_len,))
encoder_embedding = updated_classification_model.layers[1](encoder_inputs)
encoder_lstm, state_h, state_c = updated_classification_model.layers[2](encoder_embedding)

In [ ]:
output_vocab_size = 8000
target_max_len = 50

decoder_input = Input(shape=(target_max_len,))
decoder_embedding = Embedding(output_vocab_size, embedding_dim)(decoder_input)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_output, _, _ = decoder_lstm(decoder_embedding, initial_state=[state_h, state_c])
decoder_dense = Dense(output_vocab_size, activation='softmax')
decoder_output = decoder_dense(decoder_output)

In [ ]:
seq2seq_model = Model(inputs=[encoder_inputs, decoder_input], outputs=decoder_output)
seq2seq_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
encoder_input_data = x_train[:1000]
decoder_input_data = np.random.randint(1, output_vocab_size, (1000, target_max_len))
decoder_target_data = np.random.randint(1, output_vocab_size, (1000, target_max_len, 1))

In [ ]:
seq2seq_model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=32,
    epochs=1,
    validation_split=0.1
)

## **Hugging Face Tutorial**

In [ ]:
!pip install --upgrade huggingface_hub
!pip install -U datasets fsspec
!pip install -U sentence-transformers
!pip install evaluate
!pip install rouge_score
!pip install evaluate[visualization]
!pip install accelerate
!pip install  bitsandbytes
!pip install langchain-huggingface
!pip install langchain_community

In [ ]:
!huggingface-cli login
!huggingface-cli whoami

In [ ]:
!huggingface-cli repo create livehfcourse --repo-type model
!huggingface-cli repo create livehfdatacourse --repo-type dataset
!huggingface-cli repo create dummy --repo-type=space --space=docker

In [ ]:
!pip install --upgrade "huggingface_hub>=0.33.1"

In [ ]:
import os
import re
import json
from huggingface_hub import HfApi, login, notebook_login, whoami, list_repo_files, hf_hub_download
from langchain_huggingface import ChatHuggingFace
from langchain_huggingface import HuggingFaceEndpoint
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFacePipeline
from langchain_huggingface import ChatHuggingFace
from transformers import BitsAndBytesConfig
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from sentence_transformers import SentenceTransformer
from transformers import Trainer, TrainingArguments
from evaluate.visualization import radar_plot
from datasets import Dataset
from wordcloud import WordCloud
from datasets import load_dataset
from google.colab import userdata
from collections import Counter
from datasets import load_dataset
import pandas as pd
import numpy as np
import evaluate
from evaluate import load
import matplotlib.pyplot as plt


READ_TOKEN=userdata.get("HF_READ_TOKEN")
WRITE_TOKEN=userdata.get("HF_WRITE_TOKEN")

api = HfApi(token=WRITE_TOKEN)
api.create_repo(
  repo_id="Emon3412/newRepo",
  repo_type="model",
  private=False
)

api.upload_file(
    path_or_fileobj="",
    path_in_repo="config.json",
    repo_id="Emon3412/newRepo",
)

In [ ]:
login(READ_TOKEN)
login(WRITE_TOKEN)
notebook_login()

dataset = load_dataset("imbd")
print(dataset)
print(dataset["train"])
print(dataset["train"][0])
print(dataset["train"].features)
print(dataset["train"].column_names)
print(dataset["train"].features["label"].names)
print(dataset["train"].num_rows)
print(dataset["train"].num_columns)
print(dataset["train"].shape)

In [ ]:
shuffled = dataset["train"].shuffle()
shuffled

In [ ]:
shuffled = dataset["train"].shuffle(seed=42)
shuffled

In [ ]:
shuffled.select(range(100))
subset_train = dataset["train"].shuffle(seed=42).select(range(1000))

In [ ]:
short_reviews = dataset["train"].filter(lambda x: len(x["text"]) < 100)
short_reviews = subset_train.filter(lambda x: len(x["text"]) < 100)
short_positive = dataset["train"].filter(lambda x: x["label"] == 1 and len(x["text"]) < 200)
short_review, short_postive

In [ ]:
short_reviews = dataset["train"].filter(lambda x: len(x["text"]) < 100)
short_reviews = subset_train.filter(lambda x: len(x["text"]) < 100)
short_positive = dataset["train"].filter(lambda x: x["label"] == 1 and len(x["text"]) < 200)
display(short_reviews, short_positive)

In [ ]:
big_data = load_dataset("openwebtext", streaming=True)
big_data

In [ ]:
for item in big_data["train"].take(5):
  print(item)

In [ ]:
web_crawled_data = load_dataset("c4", "en", split="train", streaming=True)
for i, item in enumerate(web_crawled_data):
  print(item)
  if i > 4:
    break

In [ ]:
dataset = load_dataset("tweet_eval", "sentiment")
dataset

In [ ]:
labels = [example["label"] for example in dataset["train"]]
plt.hist(labels, bins=3, edgecolor="black")
plt.xticks([0,1,2], ["Negative", "Neutral", "Positive"])
plt.title("Tweet Sentiment Distribution")
plt.show()

In [ ]:
label_counts = Counter([ex["label"] for ex in dataset["train"]])
labels = ["Negative", "Neutral", "Positive"]
counts = [label_counts[i] for i in range(3)]
counts

In [ ]:
plt.pie(counts, labels=labels, autopct="%1.1f%%", startangle=140)
plt.title("Sentiment Proportions (Pie Chart)")
plt.axis("equal")
plt.show()

In [ ]:
all_words = []
for i in range(1000):
    words = re.findall(r"\w+", dataset["train"][i]["text"].lower())
    all_words.extend(words)

top_words = Counter(all_words).most_common(20)
words, counts = zip(*top_words)
plt.figure(figsize=(10,5))
plt.bar(words, counts, color="orange")
plt.xticks(rotation=45)
plt.title("Top 20 Most Common Words in Tweets")
plt.tight_layout()
plt.show()

In [ ]:
all_text = " ".join(dataset["train"][i]["text"] for i in range(1000))
wordcloud = WordCloud(width=800, height=400, background_color="white").generate(all_text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Most Frequent Words in Tweets")
plt.show()

In [ ]:
for label_id, label_name in enumerate(["Negative", "Neutral", "Positive"]):
  text = " ".join(example["text"] for example in dataset["train"] if example["label"] == label_id)
  wordcloud = WordCloud(width=800, height=400).generate(text)

  plt.figure(figsize=(10, 5))
  plt.imshow(wordcloud, interpolation="bilinear")
  plt.axis("off")
  plt.title(f"Most Frequent Words in {label_name} Tweets")
  plt.show()

In [ ]:
data = {
    "text": [
        "I love this product! Works great ",
        "Terrible experience, I want a refund.",
        "Fast delivery and excellent packaging.",
        "Worst app I’ve ever used. Crashed in 1 minute.",
        "Super helpful support team, thanks a lot!",
        "Nothing worked. Waste of time."
    ],
    "label": [1, 0, 1, 0, 1, 0]
}

df = pd.DataFrame(data)
dataset = Dataset.from_pandas(df)
dataset = dataset.class_encode_column("label")
dataset

In [ ]:
os.environ["HF_TOKEN"] = WRITE_TOKEN
dataset.push_to_hub("Emon3412/newRepo")

notebook_login()
print(whoami())
dataset.push_to_hub("Emon3412/newRepo")

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, AutoModelForCausalLM
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
from torch.nn.functional import cosine_similarity
from transformers import PreTrainedTokenizerFast
from huggingface_hub import snapshot_download, InferenceClient
from transformers import pipeline


def tokenize(example):
  return tokenizer(example["text"], padding="max_length", truncation=True)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenize(dataset["train"][0])
encoded_dataset = dataset.map(tokenize, batched=True)
encoded_dataset, encoded_dataset['input_ids'][0], encoded_dataset['token_type_ids'][0]

In [ ]:
tokens = tokenizer.tokenize("Hello, how are you?")
tokens

In [ ]:
text = ["This is a sample sentence for tokenization."] * 100000
start = time.time()
tok_fast = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=True)
tok_fast(text, padding=True, truncation=True)
print("Fast tokenizer time:", time.time() - start)

In [ ]:
start = time.time()
tok_slow = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=False)
tok_slow(text, padding=True, truncation=True)
print("Slow tokenizer time:", time.time() - start)

In [ ]:
corpus = ["I love transformers!", "Tokenizers are amazing.", "Build your own.\n"]
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(vocab_size=100)
tokenizer.train_from_iterator(corpus, trainer)
tokenizer.save("custom-tokenizer.json")

In [ ]:
hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="/content/custom-tokenizer.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]"
)
hf_tokenizer.save_pretrained("my_tokenizer_hf")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
inputs = tokenizer("Hello, how are you?", return_tensors="pt")
model = AutoModel.from_pretrained("bert-base-cased")
with torch.no_grad():
  outputs = model(**inputs)


print(outputs)
print(outputs.last_hidden_state.shape)
print(outputs.last_hidden_state[0])
print(len(outputs.last_hidden_state[0][0]))
print(outputs.last_hidden_state[0][1])
len(outputs.last_hidden_state[0].mean(dim=0))

In [ ]:
sentence1 = "Hello, how are you?"
sentence2 = "Hi, how do you do?"

inputs1 = tokenizer(sentence1, return_tensors="pt")
inputs2 = tokenizer(sentence2, return_tensors="pt")

with torch.no_grad():
  outputs1 = model(**inputs1).last_hidden_state[0].mean(dim=0)
  outputs2 = model(**inputs2).last_hidden_state[0].mean(dim=0)

similarity = cosine_similarity(outputs1, outputs2, dim=0)
print(f"Similarity between '{sentence1}' and '{sentence2}': {similarity.item():.4f}")

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embedding1 = model.encode(sentence1, convert_to_tensor=True)
embedding2 = model.encode(sentence2, convert_to_tensor=True)

similarity = cosine_similarity(embedding1, embedding2, dim=0)
print(f"Similarity between '{sentence1}' and '{sentence2}': {similarity.item():.4f}")

### Example of AutoModelClass with Different heads

| Task | Class Name | Description |
| :--- | :--- | :--- |
| **Base Model** | `AutoModel` | Just embeddings (no head), for feature extraction |
| **Masked LM** | `AutoModelForMaskedLM` | For BERT-style `[MASK]` prediction |
| **Sequence Classification** | `AutoModelForSequenceClassification` | For sentence-level tasks like sentiment, spam |
| **Token Classification** | `AutoModelForTokenClassification` | For NER, POS tagging (token-level labels) |
| **Multiple Choice** | `AutoModelForMultipleChoice` | For MCQ tasks (e.g., SWAG dataset) |
| **Next Sentence Prediction** | `AutoModelForNextSentencePrediction` | For NSP (mainly BERT pretraining) |
| **Causal LM (Text Gen)** | `AutoModelForCausalLM` | For GPT-style models (left-to-right text generation) |
| **Seq2Seq LM** | `AutoModelForSeq2SeqLM` | For translation, summarization (T5, BART) |
| **Question Answering** | `AutoModelForQuestionAnswering` | For extractive QA (SQuAD-style) |
| **Vision Tasks** | `AutoModelForImageClassification`, etc. | For image classification (e.g., ViT) |
| **Audio Tasks** | `AutoModelForAudioClassification`, etc. | For Wav2Vec, Whisper, etc. |
| **Conditional Generation** | `AutoModelForConditionalGeneration` | For T5/BART-style seq2seq tasks |
| **Zero-shot Tasks** | `AutoModelForZeroShotClassification` (via pipeline) | For inference without fine-tuning |
| **Contrastive Learning** | `AutoModelForContrastiveLearning` | For embeddings comparison tasks |

In [ ]:
sentence = "I am doing very happy"
inputs = tokenizer(sentence, return_tensors="pt")
inputs.pop("token_type_ids")
inputs

In [ ]:
clf_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

with torch.no_grad():
  outputs = clf_model(**inputs)

logits = outputs.logits
predicted_class = torch.argmax(logits, dim=1)
print("Predicted Class:", predicted_class.item())

In [ ]:
gpt_tok = AutoTokenizer.from_pretrained("gpt2")
gpt_model = AutoModelForCausalLM.from_pretrained("gpt2")
sentence = "Transformers are beast"
config = AutoConfig.from_pretrained("bert-base-uncased")
config.num_labels = 5
gpt_inputs = gpt_tok(sentence, return_tensors="pt")

with torch.no_grad():
  gpt_output = gpt_model.generate(
      gpt_inputs["input_ids"],
      max_length=gpt_inputs["input_ids"].shape[1] + 5,
      do_sample=False
  )

generated_text = gpt_tok.decode(gpt_output[0], skip_special_tokens=True)
print("Generated Text:", generated_text)
print(config.hidden_size)
print(config.num_attention_heads)
print(config.vocab_size)
print(config.hidden_act)
print(config.num_labels)

In [ ]:
model = AutoModel.from_config(config)
model

In [ ]:
snapshot_download(
    repo_id="bert-base-uncased",
    local_dir="/content/test",
    local_dir_use_symlinks=False
)
tokenizer = AutoTokenizer.from_pretrained("/content/test")
model = AutoModel.from_pretrained("/content/test")
model

| Parameter | Type | Description / Use Case |
| :--- | :--- | :--- |
| `task` | `str` | Task name like `"text-classification"`, `"text-generation"`, `"summarization"`, etc. |
| `model` | `str` or `PreTrainedModel` | Model name (e.g., `"bert-base-uncased"`) or loaded model object |
| `tokenizer` | `str` or `Tokenizer Object` | Optional – auto-loaded from model if not given |
| `config` | `str` or `PretrainedConfig` | Optional – custom configuration (e.g., num labels, dropout, etc.) |
| `framework` | `str` (`"pt"` or `"tf"`) | Force PyTorch or TensorFlow (auto-detected if not set) |
| `device` | `int` or `str` | `0` = GPU, `-1` = CPU, `"cuda"` or `"cpu"` |
| `revision` | `str` | Git branch/tag/commit from Hugging Face Hub (e.g., `"main"`, `"v1.0"`) |
| `use_fast` | `bool` | Whether to use fast tokenizer (defaults to `True`) |
| `token` | `str` or `bool` | Hugging Face token (needed for private models or rate limits) |
| `device_map` | `Any` | For multi-GPU / model parallelism |
| `torch_dtype` | `torch.dtype` | Force precision: e.g., `torch.float16` for faster inference |
| `trust_remote_code` | `bool` | Allow loading custom `model.py` logic from HF repo (for advanced/custom models) |
| `model_kwargs` | `dict` | Extra arguments to pass to model (e.g., `temperature`, `top_p`, `max_length`, etc.) |
| `feature_extractor` | `str` or `Object` | Used for older vision/audio models (now replaced by `image_processor`) |
| `image_processor` | `BaseImageProcessor` | For image tasks (used in ViT, SAM, etc.) |
| `processor` | `ProcessorMixin` | For multi-modal models (e.g., Whisper, CLIP, LayoutLMv3) |
| `pipeline_class` | `Any` | Custom pipeline class if you're extending or modifying the base pipeline logic |
| `**kwargs` | `Any` | Additional pipeline-specific keyword arguments (like `max_length`, `top_k`, etc.) |

In [ ]:
classifier=pipeline("sentiment-analysis")
classifier("i will learn AI throughout my entire life it is like a passion")
classifier("mera dost don hai")
result = pipeline("sentiment-analysis")("Hugging Face is awesome")
lassifier=pipeline("zero-shot-classification")
res = lassifier(
    "This is a course about Python list comprehension",
    candidate_labels=["education", "politics", "business"],
)

generation=pipeline("text-generation")
generation("python is a simple language what is your thought?")
generation("hello how are you?")
summarizer=pipeline("summarization")

text="A large language model (LLM) is a machine learning model that uses deep learning to perform natural language processing (NLP) tasks. LLMs are trained on large amounts of text to learn how to respond to user requests with human-like language."

print(result)
print(res)
print(summarizer(text,max_length=10))

In [ ]:
summarizer = pipeline("summarization", model="google/long-t5-tglobal-base")
summary = summarizer(text, max_length=50, min_length=10, do_sample=False)
qa = pipeline("question-answering")

texts = ["I absolutely love transformers!"] * 100

start = time.time()
bert = pipeline("sentiment-analysis", model="bert-base-uncased")
bert(texts)
print("BERT Time:", time.time() - start)

start = time.time()
distil = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
distil(texts)
print("DistilBERT Time:", time.time() - start)

print(summary[0]["summary_text"])
print(qa(question="Where do I work?", context="I work at Hugging Face."))

In [ ]:
accuracy = accuracy.load("accuracy")
result = accuracy.compute(predictions=[0, 1, 1, 0], references=[0, 1, 0, 0])
print(result)

In [ ]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  return accuracy.compute(predictions=predictions, references=labels)


def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

train_dataset = load_dataset("imdb", split="train[:100]")
test_dataset = load_dataset("imdb", split="test[:100]")

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

train_encoded = train_dataset.map(tokenize, batched=True)
test_encoded = test_dataset.map(tokenize, batched=True)
args = TrainingArguments(
    output_dir="eval_check",
    per_device_eval_batch_size=32,
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=args,
    compute_metrics=compute_metrics,
)
metrics = trainer.evaluate(eval_dataset=test_encoded)
print(metrics)

In [ ]:
bleu = load("bleu")
predictions = ["The cat is sitting on the mat"]
references = [["The cat sat on the mat"]]

result = bleu.compute(predictions=predictions, references=references)
print("BLEU Score: ", result)

In [ ]:
predictions = ["Government unveils new climate action plan"]
references = [["Government announces new plan to fight climate change"]]

result = bleu.compute(predictions=predictions, references=references)
print("BLEU Score (Text Gen):", result)

In [ ]:
rouge = load("rouge")

predicted_summary = ["Hugging Face develops powerful AI tools."]
reference_summary = ["Hugging Face is a company that builds AI models."]

result = rouge.compute(predictions=predicted_summary, references=reference_summary)
print("Perplexity Score:", result)

In [ ]:
perplexity = load("perplexity")

predictions = ["The government announced a new climate policy today."]

reslt = perplexity.compute(predictions=predictions, model_id="gpt2")
print("Perplexity Score:", result)

In [ ]:
api = HfApi()
model_info = api.model_info(model="bert-base-uncased")
models = api.list_models(author="bert-base-uncased")

print("Model Card Metadata:", model_info.card_data)
print("Tags:", model_info.tags)
print("Commit SHA:", model_info.sha)
print("Last Modified:", model_info.last_modified)
print("Files:", [f.rfilename for f in model_info.siblings])
print("Model ID:", model_info.card_data.get('modelId', 'Not available'))
print("Likes:", model_info.likes)
print("Author/Namespace:", model_info.author)
print("Is Private:", model_info.private)
print("Downloads:", model_info.downloads)

In [ ]:
models = api.list_models(search="emotion", sort="downloads", limit=5)
for model in models:
  print(model.modelId, "-", model.downloads)

In [ ]:
models = api.list_models(filter="text-generation", sort="downloads", limit=10)

for m in models:
    print(m.modelId, "-", m.downloads)

In [ ]:
files = list_repo_files("google/flan-t5-base")
dataset = api.dataset_info("imdb")
config_path = hf_hub_download(repo_id="bert-base-uncased", filename="config.json")
client = InferenceClient(model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
response = client.text_classification("I love Hugging Face! ")

with open(config_path) as f:
  config = json.load(f)

print(files)
print(response)
print("Pipeline tag: ", info.pipeline_tag)
print("Description:", dataset.cardData.get("summary", "No summary"))
print("Files:", [f.rfilename for f in dataset.siblings])
print("Hidden size:", config["hidden_size"])

In [ ]:
models = api.list_models(filter="text-generation", limit=20)
print("Sample models with inference support:")
for m in models:
    if m.pipeline_tag == "text-generation":
        print("-", m.modelId)

In [ ]:
gen = api.list_models(
    inference_provider="hf-inference",
    pipeline_tag="text-generation",
    limit=25
)

models = list(gen)
for model_name in models:
  print(model_name.modelId)

In [ ]:
model_id = models[0].modelId
client = InferenceClient(model="meta-llama/Llama-3.1-8B-Instruct", provider="hf-inference")

out = client.text_generation(
        prompt="I saw a puppy, a cat and a raccoon during my bike ride in the park.",
        max_new_tokens=80,
        temperature=0.7)
print(out)

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-R1",
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
)

chat_model = ChatHuggingFace(llm=llm)
chat_mode.invoke("hello how are you?")

In [ ]:
template="""Question: {question}
give me a answer in detail manner and in step by step manner"""
prompt=PromptTemplate(template=template,input_variables=["question"])
llm_chain = (
        {"question": RunnablePassthrough()}
        | prompt
        | chat_model
        | StrOutputParser()
    )

question="who is a first president of BANGLADESH?"
llm_chain.invoke({"question":question})

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

llm = HuggingFacePipeline.from_model_id(
    model_id="HuggingFaceH4/zephyr-7b-beta",
    task="text-generation",
    pipeline_kwargs=dict(
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.03,
        return_full_text=False,
    ),
    model_kwargs={"quantization_config": quantization_config},
)

chat_model = ChatHuggingFace(llm=llm)
template="""Question: {question}
give me a answer in detail manner and in step by step manner"""
prompt=PromptTemplate(template=template,input_variables=["question"])

llm_chain = (
        {"question": RunnablePassthrough()}
        | prompt
        | chat_model
        | StrOutputParser()
    )

question="who is a first president of BANGLADESH?"
llm_chain.invoke({"question":question})

### **Bert Fine-Tuning**

In [ ]:
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer,
    BertTokenizerFast,
    BertForSequenceClassification,
    BertForTokenClassification,
    BertForQuestionAnswering,
    get_linear_schedule_with_warmup
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score
import numpy as np
from tqdm import tqdm
from torch.optim import AdamW

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

dataset = load_dataset("imdb")
dataset

In [ ]:
train_dataset = dataset["train"].select(range(1000))
test_dataset = dataset["test"].select(range(500))
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

def preprocess(ds):
  ds = ds.map(tokenize_fn, batched=True, remove_columns=["text"])
  ds = ds.rename_column("label", "labels")
  ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
  return ds

train_dataset = preprocess(train_dataset)
test_dataset = preprocess(test_dataset)

In [ ]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

for layer in model.bert.encoder.layer:
  print(layer)

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert-finetuned-imdb",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    logging_dir="./logs",
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

In [ ]:
!tensorboard --logdir=./logs

In [ ]:
trainer.save_model("./bert-finetuned-imdb")

In [ ]:
tokenizer.save_pretrained("./bert-finetuned-imdb")
metrics = trainer.evaluate()
print(metrics)

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("./bert-finetuned-imdb")
classifer = pipeline("text-classification", model=model, tokenizer=tokenizer)

text = "This movie was amazing and I loved the acting!"
result = classifer(text)
print(result)

In [ ]:
notebook_login()
print(whoami())
tokenizer.push_to_hub("Emon3469/bert-finetuned-imdb")
trainer.push_to_hub("Emon3469/bert-finetuned-imdb")

In [ ]:
class Basket:
  def __init__(self, fruits):
    self.fruits = fruits

  def __len__(self):
    return len(self.fruits)

  def __getitem__(self, index):
    return self.fruits[index]

basket = Basket(["Apple", "Banana", "Mango"])
print(len(basket))
print(basket[0])
print(basket[0])

In [ ]:
class TextClassificationDataset(Dataset):
  def __init__(self, texts, labels, tokenizer, max_length=512):
    self.texts = texts
    self.labels = labels
    self.tokenizer = tokenizer
    self.max_length = max_length

  def __len__(self):
    return len(self.texts)

  def __getitem__(self, idx):
    text = str(self.texts[idx])
    label = self.labels[idx]

    encoding = self.tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=self.max_length,
        return_tensors="pt"
    )

    return {
        'input_ids': encoding['input_ids'].flatten(),
        'attention_mask': encoding['attention_mask'].flatten(),
        'labels': torch.tensor(label, dtype=torch.long)
    }

In [ ]:
class BERTTextClassifier:
  def __init__(self, model_name='bert-base-uncased', num_classes=2, max_length=512):
    self.model_name = model_name
    self.num_classes = num_classes
    self.max_length = max_length

    self.tokenizer = BertTokenizerFast.from_pretrained(model_name)
    self.model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_classes)
    self.model.to(device)

  def load_imdb_data(self, sample_size=5000):
    dataset = load_dataset("imdb")
    train_indices = np.random.choice(len(dataset['train']), size=sample_size, replace=False)
    test_indices = np.random.choice(len(dataset['test']), size=sample_size, replace=False)

    train_texts = [dataset['train'][int(i)]['text'] for i in train_indices]
    train_labels = [dataset['train'][int(i)]['label'] for i in train_indices]

    test_texts = [dataset['test'][int(i)]['text'] for i in test_indices]
    test_labels = [dataset['test'][int(i)]['label'] for i in test_indices]

    print(f

In [ ]:
class NERDataset(Dataset):
  def __init__(self, tokens_list, labels_list, tokenizer, max_length=512):
    self.tokens_list = tokens_list
    self.labels_list = labels_list
    self.tokenizer = tokenizer
    self.max_length = max_length

  def __len__(self):
    return len(self.tokens_list)

  def __getitem__(self, idx):
    tokens = self.tokens_list[idx]
    labels = self.labels_list[idx]

    encoding = self.tokenizer(
        tokens,
        truncation=True,
        padding="max_length",
        max_length=self.max_length,
        is_split_into_words=True,
        return_tensors="pt"
    )

    word_ids = encoding.word_ids(batch_index=0)
    aligned_labels = []
    previous_word_idx = None

    for word_idx in word_ids:
      if word_idx is None:
        aligned_labels.append(-100)
      elif word_idx != previous_word_idx:
        aligned_labels.append(labels[word_idx] if word_idx < len(labels) else 0)
      else:
        aligned_labels.append(aligned_labels[-1])
      previous_word_idx = word_idx

    if len(aligned_labels) < self.max_length:
      aligned_labels += [-100] * (self.max_length - len(aligned_labels))
    elif len(aligned_labels) > self.max_length:
      aligned_labels = aligned_labels[:self.max_length]

    return {
        'input_ids': encoding['input_ids'].squeeze(0),
        'attention_mask': encoding['attention_mask'].squeeze(0),
        'labels': torch.tensor(aligned_labels, dtype=torch.long)
    }

In [ ]:
class BERTNERClassifier:
  def __init__(self, model_name='bert-base-uncased', num_classes=9, max_length=512):
    self.model_name = model_name
    self.num_classes = num_classes
    self.max_length = max_length

    self.tokenizer = BertTokenizerFast.from_pretrained(model_name)
    self.model = BertForTokenClassification.from_pretrained(model_name, num_labels=num_classes)
    self.model.to(device)

    self.labels = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

  def load_wikiann_data(self, sample_size=1000):
    dataset = load_dataset("wikiann", "en")

    train_indices = np.random.choice(len(dataset['train']),
                                   min(sample_size, len(dataset['train'])),
                                   replace=False)

    test_indices = np.random.choice(len(dataset['test']),
                                  min(sample_size//4, len(dataset['test'])),
                                  replace=False)

    train_tokens = [dataset['train'][int(i)]['tokens'] for i in train_indices]
    train_labels = [dataset['train'][int(i)]['ner_tags'] for i in train_indices]

    test_tokens = [dataset['test'][int(i)]['tokens'] for i in test_indices]
    test_labels = [dataset['test'][int(i)]['ner_tags'] for i in test_indices]

    print(f"Train samples: {len(train_tokens)}")
    print(f"Test samples: {len(test_tokens)}")

    return train_tokens, train_labels, test_tokens, test_labels

  def train(self, train_tokens, train_labels, epochs=1, batch_size=8, learning_rate=2e-5):
    train_dataset = NERDataset(
            train_tokens, train_labels, self.tokenizer, self.max_length
        )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    optimizer = AdamW(self.model.parameters(), lr=learning_rate, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=total_steps//10, num_training_steps=total_steps
    )

    self.model.train()

    for epoch in range(epochs):
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')

        for batch in progress_bar:
            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            total_loss += loss.item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            progress_bar.set_postfix({'Loss': f'{loss.item():.4f}'})

        print(f'Epoch {epoch+1}, Average Loss: {total_loss/len(train_loader):.4f}')

  def evaluate(self, test_tokens, test_labels, batch_size=8):
    test_dataset = NERDataset(test_tokens, test_labels, self.tokenizer, self.max_length)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    self.model.eval()
    predictions = []
    true_labels = []

    with torch.no_grad():
      for batch in tqdm(test_loader, desc='Evaluating'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=2).cpu().numpy()
        labels = labels.cpu().numpy()

        for i in range(preds.shape[0]):
          for j in range(preds.shape[1]):
            if labels[i][j] != -100:
              predictions.append(preds[i][j])
              true_labels.append(labels[i][j])

    accuracy = accuracy_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions, average='weighted')

    return accuracy, f1

  def predict(self, tokens_list):
    predictions = []

    self.model.eval()

    for tokens in tokens_list:
      encoding = self.tokenizer(
          tokens,
          truncation=True,
          padding="max_length",
          max_length=self.max_length,
          is_split_into_words=True,
          return_tensors="pt"
      )

      input_ids = encoding['input_ids'].to(device)
      attention_mask = encoding['attention_mask'].to(device)

      with torch.no_grad():
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=2).cpu().numpy()[0]

        word_ids = encoding.word_ids(batch_index=0)
        token_predictions = []
        previous_word_idx = None

        for i, word_idx in enumerate(word_ids):
          if word_idx is not None and word_idx != previous_word_idx:
            if word_idx < len(tokens):
              token_predictions.append(self.labels[preds[i]])
          previous_word_idx = word_idx

        predictions.append(token_predictions)

    return predictions

In [ ]:
class QADataset(Dataset):
  def __init__(self, questions, contexts, answers, tokenizer, max_length=512):
    self.contexts = contexts
    self.questions = questions
    self.answers = answers
    self.tokenizer = tokenizer
    self.max_length = max_length

  def __len__(self):
    return len(self.contexts)

  def __getitem__(self, idx):
    question = self.questions[idx]
    context = self.contexts[idx]
    answer = self.answers[idx]

    encoding = self.tokenizer(
        question,
        context,
        truncation=True,
        padding="max_length",
        max_length=self.max_length,
        return_offsets_mapping=True,
        return_tensors="pt"
    )

    offset_mapping = encoding.pop('offset_mapping')[0]
    start_positions = torch.tensor(0, dtype=torch.long)
    end_positions = torch.tensor(0, dtype=torch.long)

    if answer and 'answer_start' in answer and len(answer['answer_start']):
      answer_start = answer['answer_start'][0]
      answer_text = answer['text'][0]
      answer_end = answer_start + len(answer_text)

      for i, (start, end) in enumerate(offset_mapping):
        if start <= answer_start < end:
          start_positions = torch.tensor(i, dtype=torch.long)
        if start <= answer_end < end:
          end_positions = torch.tensor(i, dtype=torch.long)
          break

    return {
        'input_ids': encoding['input_ids'].squeeze(0),
        'attention_mask': encoding['attention_mask'].squeeze(0),
        'start_positions': start_positions,
        'end_positions': end_positions
      }

In [ ]:
class BERTQuestionAnswering:
  def __init__(self, model_name='bert-base-uncased', max_length=512):
    self.model_name = model_name
    self.max_length = max_length

    self.tokenizer = BertTokenizerFast.from_pretrained(model_name)
    self.model = BertForQuestionAnswering.from_pretrained(model_name)
    self.model.to(device)

  def load_squad_data(self, sample_size=2000):
    dataset = load_dataset("squad")
    train_indices = np.random.choice(len(dataset['train']),
                                       min(sample_size, len(dataset['train'])),
                                       replace=False)
    val_indices = np.random.choice(len(dataset['validation']),
                                     min(sample_size//4, len(dataset['validation'])),
                                     replace=False)

    train_questions = [dataset['train'][int(i)]['question'] for i in train_indices]
    train_contexts = [dataset['train'][int(i)]['context'] for i in train_indices]
    train_answers = [dataset['train'][int(i)]['answers'] for i in train_indices]

    val_questions = [dataset['validation'][int(i)]['question'] for i in val_indices]
    val_contexts = [dataset['validation'][int(i)]['context'] for i in val_indices]
    val_answers = [dataset['validation'][int(i)]['answers'] for i in val_indices]

    print(f"Train samples: {len(train_questions)}")
    print(f"Validation samples: {len(val_questions)}")

    return (train_questions, train_contexts, train_answers,
            val_questions, val_contexts, val_answers)

  def train(self, questions, contexts, answers, epochs=1, batch_size=8, learning_rate=2e-5):
    train_dataset = QADataset(questions, contexts, answers, self.tokenizer, self.max_length)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    optimizer = AdamW(self.model.parameters(), lr=learning_rate, weight_decay=0.01)

    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
          optimizer, num_warmup_steps=total_steps//10, num_training_steps=total_steps
    )

    self.model.train()

    for epoch in range(epochs):
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')

        for batch in progress_bar:
          optimizer.zero_grad()

          input_ids = batch['input_ids'].to(device)
          attention_mask = batch['attention_mask'].to(device)
          start_positions = batch['start_positions'].to(device)
          end_positions = batch['end_positions'].to(device)

          outputs = self.model(
              input_ids=input_ids,
              attention_mask=attention_mask,
              start_positions=start_positions,
              end_positions=end_positions
            )

          loss = outputs.loss
          total_loss += loss.item()
          loss.backward()

          torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
          optimizer.step()
          scheduler.step()
          progress_bar.set_postfix({'Loss': f'{loss.item():.4f}'})

        print(f'Epoch {epoch+1}, Average Loss: {total_loss/len(train_loader):.4f}')

  def answer_question(self, question, context, max_answer_len=30):
    encoding = self.tokenizer(
        question,
        context,
        truncation=True,
        padding='max_length',
        max_length=self.max_length,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    self.model.eval()

    with torch.no_grad():
      outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)

      start_logits = outputs.start_logits
      end_logits = outputs.end_logits
      start_idx = torch.argmax(start_logits, dim=1).item()
      end_idx = torch.argmax(end_logits, dim=1).item()

      if end_idx < start_idx:
        end_idx = start_idx
      if (end_idx - start_idx) > max_answer_len:
        end_idx = start_idx + max_answer_len

      answer_tokens = input_ids[0][start_idx:end_idx+1]
      answer = self.tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()

      return answer

In [ ]:
def run_text_classification_demo():
  print("\n" + "="*60)
  print("TEXT CLASSIFICATION (Sentiment Analysis) DEMO")
  print("="*60)

  classifier = BERTTextClassifier()
  train_texts, train_labels, test_texts, test_labels = classifier.load_imdb_data(sample_size=1000)

  print(f"\nSample Review: {train_texts[0][:200]}...")
  print(f"Label: {'Positive' if train_labels[0] == 1 else 'Negative'}")

  classifier.train(train_texts, train_labels, epochs=1, batch_size=8)
  accuracy, f1, report = classifier.evaluate(test_texts, test_labels, batch_size=8)

  print(f"\nAccuracy: {accuracy:.4f}")
  print(f"F1 Score: {f1:.4f}")

  custom_reviews = [
    "This movie was fantastic! Amazing acting and great plot.",
    "Boring and terrible. Waste of time.",
    "Not bad, could be better though."
  ]

  predictions, probabilities = classifier.predict(custom_reviews)

  print(f"\nCustom Predictions:")
  for text, pred, prob in zip(custom_reviews, predictions, probabilities):
    sentiment = "Positive" if pred == 1 else "Negative"
    confidence = prob[pred] * 100
    print(f"'{text[:50]}...' -> {sentiment} ({confidence:.1f}%)")

In [ ]:
def run_ner_demo():
    print("\n" + "="*60)
    print("NAMED ENTITY RECOGNITION DEMO")
    print("="*60)

    ner_model = BERTNERClassifier(num_labels=9)
    train_tokens, train_labels, test_tokens, test_labels = ner_model.load_wikiann_data(sample_size=500)

    print(f"\nSample tokens: {train_tokens[0][:10]}")

    label_names = [ner_model.labels[l] if l < len(ner_model.labels) else "O" for l in train_labels[0][:10]]

    print(f"Sample labels: {label_names}")

    ner_model.train(train_tokens, train_labels, epochs=1, batch_size=8)
    accuracy, f1 = ner_model.evaluate(test_tokens, test_labels, batch_size=8)
    print(f"\nAccuracy: {accuracy:.4f}")
    print(f"F1 Score: {f1:.4f}")
    custom_sentences = [
        ["John", "Smith", "works", "at", "Google", "in", "California"],
        ["Apple", "Inc.", "was", "founded", "by", "Steve", "Jobs"]
    ]

    predictions = ner_model.predict(custom_sentences)
    print(f"\nCustom Predictions:")
    for tokens, preds in zip(custom_sentences, predictions):
        print("Tokens:", tokens)
        print("Labels:", preds)
        print()

In [ ]:
def run_qa_demo():
    print("\n" + "="*60)
    print("QUESTION ANSWERING DEMO")
    print("="*60)

    qa_model = BERTQuestionAnswering()
    (train_questions, train_contexts, train_answers,
     val_questions, val_contexts, val_answers) = qa_model.load_squad_data(sample_size=500)
    ans_text = train_answers[0]['text'][0] if train_answers[0]['text'] else 'No answer'

    print(f"\nSample Question: {train_questions[0]}")
    print(f"Sample Context: {train_contexts[0][:200]}...")
    print(f"Sample Answer: {ans_text}")

    qa_model.train(train_questions, train_contexts, train_answers, epochs=1, batch_size=4)
    print(f"\nCustom Q&A Examples:")

    test_cases = [
        {
            "question": "What is the capital of France?",
            "context": "France is a country in Europe. Paris is the capital and largest city of France. The city is known for the Eiffel Tower and the Louvre Museum."
        },
        {
            "question": "Who founded Apple?",
            "context": "Apple Inc. is an American technology company. It was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in 1976. The company is known for products like iPhone and Mac."
        }
    ]

    for case in test_cases:
        answer = qa_model.answer_question(case["question"], case["context"], max_answer_len=30)
        print(f"Q: {case['question']}")
        print(f"A: {answer}")
        print()

In [ ]:
print("BERT Multi-Task Demo")
print("Choose a task to run:")
print("1. Text Classification (Sentiment Analysis)")
print("2. Named Entity Recognition (NER)")
print("3. Question Answering")
print("4. Run All Tasks")

choice = input("\nEnter your choice (1-4): ").strip()

try:
    if choice == "1":
        run_text_classification_demo()
    elif choice == "2":
        run_ner_demo()
    elif choice == "3":
        run_qa_demo()
    elif choice == "4":
        run_text_classification_demo()
        run_ner_demo()
        run_qa_demo()
    else:
        print("Invalid choice! Please run again.")
except Exception as e:
    print("\n--- ERROR OCCURRED ---")
    print(f"Error: {e}")
    print("\nMake sure you have:")
    print("1. Installed required packages:")
    print("   pip install torch transformers datasets scikit-learn tqdm numpy")
    print("2. Loaded all classes & dataset helpers (BERTTextClassifier, BERTNERClassifier, BERTQuestionAnswering, TextClassificationDataset, NERDataset, QADataset)")
    print("3. Using the fixed versions (with int casting, offset_mapping for QA, word_ids fix for NER)")